# 6. Forecast validation

**Stage 1, Step 5 — sections 29, 32, 39.**

The checks that decide whether this system can be trusted, as opposed to whether it runs:

1. **Leakage** — including a deliberate demonstration that the detector fires.
2. **Train/serve equivalence** — training and serving build features by different paths, which is new in this step.
3. **The refusal behaviour** — what happens past the end of the planning calendar.
4. **The agent contract** — what a Claude agent actually receives.

In [ ]:
from __future__ import annotations

import sys
import warnings
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

from app.services.container import Container
from ml.forecasting.baselines import attach_seasonal_reference
from ml.forecasting.config import load_forecast_config
from ml.forecasting.dataset import (
    ORIGIN_DATE,
    TARGET_DATE,
    TARGET_PREFIX,
    HorizonDataset,
    build_history,
    build_horizon_dataset,
    target_side_features,
)
from ml.forecasting.sampling import sample_series

repo = Container().data_repository
config = load_forecast_config().smoke()

sample = sample_series(repo, n_series=config.sampling.n_series, seed=config.sampling.seed)
history = build_history(repo, config, sample)
view = repo.as_of(pd.to_datetime(history["date"]).dt.date.max())

## 1. The mutation test, and proof that it bites

Corrupt every `OBSERVED` value after a cutoff, rebuild, and check that features for origins *before* the cutoff are byte-identical.

Two details make this test meaningful rather than decorative:

- **`calendar`, `promotions` and `pricing` are deliberately left alone.** They are `KNOWN_IN_ADVANCE`, so reading them forward is legitimate and this design does exactly that. Corrupting them too would assert a property the system does not have and should not have.
- **The target is excluded from the comparison.** It lives at `origin + h`, which is legitimately after the cutoff, so it *should* change.

In [ ]:
origins = pd.to_datetime(history["date"]).dt.date
cutoff = origins.min() + pd.Timedelta(days=(origins.max() - origins.min()).days // 2)
cutoff = cutoff.date() if hasattr(cutoff, "date") else cutoff

clean = build_horizon_dataset(history, view, config, sample, seed=5)

corrupted_history = history.copy()
after = pd.to_datetime(corrupted_history["date"]).dt.date > cutoff
for column in ("units", "lag_1_units", "rolling_7_units"):
    if column in corrupted_history.columns:
        corrupted_history.loc[after, column] *= 1000

corrupted = build_horizon_dataset(corrupted_history, view, config, sample, seed=5)

features = [c for c in clean.feature_names if c != "units"]
mask_clean = pd.to_datetime(clean.frame[ORIGIN_DATE]).dt.date <= cutoff
mask_corrupt = pd.to_datetime(corrupted.frame[ORIGIN_DATE]).dt.date <= cutoff

a = clean.frame[mask_clean][features].reset_index(drop=True)
b = corrupted.frame[mask_corrupt][features].reset_index(drop=True)

print(f"cutoff                     : {cutoff}")
print(f"pre-cutoff rows compared   : {len(a):,}")
print(f"features identical         : {a.equals(b)}")

In [ ]:
# Now plant the exact bug the design prevents: source target-side features from
# the ORIGIN date while labelling them as the target's.
leaky = build_horizon_dataset(
    history, view, config, sample, seed=5, horizon_features_from_target=True
)

column = f"{TARGET_PREFIX}day_of_week"
honest_values = clean.frame[column].reset_index(drop=True)
leaky_values = leaky.frame[column].reset_index(drop=True)

print(f"honest and leaky agree on {column}: {honest_values.equals(leaky_values)}")
print()
print("They must NOT agree. Without this demonstration the test above is")
print("unfalsifiable - a test that has never failed proves nothing about its")
print("ability to detect anything.")

## 2. Train/serve equivalence

New in this step. Training reads target-side features over historical dates; serving reads them over future ones. Two code paths computing what must be one thing is the classic source of silent skew, so both go through `target_side_features`.

In [ ]:
pairs = sample.pairs.head(4)

narrow = target_side_features(view, pairs, pd.date_range("2025-06-01", periods=7, freq="D"))
wide = target_side_features(view, pairs, pd.date_range("2024-01-01", periods=400, freq="D"))

print(f"columns match between a 7-day and a 400-day window: {set(narrow.columns) == set(wide.columns)}")
print()
for col in (f"{TARGET_PREFIX}days_to_festival", f"{TARGET_PREFIX}days_since_festival"):
    print(f"  {col:32s} present in narrow window: {col in narrow.columns}")

This one caught a real defect. `add_festival_proximity` measures distance to the nearest festival *in whatever calendar it is handed*. Over a full history that is plentiful; over a 7-day serving window there may be none, and the columns came out missing entirely — so the model raised at serving time on features it had trained with.

Both paths now read a wide calendar window regardless of how many days are being forecast.

## 3. The refusal behaviour (§29)

In [ ]:
from app.schemas.domain import ForecastHorizon
from app.schemas.forecast import ForecastErrorResponse, ForecastRequest
from app.services.forecast_service import ForecastingService

model_dir = Path.cwd().parents[1] / "data" / "local" / "models" / "forecasting_sampled"
if not (model_dir / "model.joblib").is_file():
    model_dir = Path.cwd().parents[1] / "data" / "local" / "models" / "forecasting"

service = ForecastingService(repo, model_dir=model_dir)
print("model available:", service.is_available)

if service.is_available:
    pair = service.model.pairs.iloc[0]

    ok = service.forecast(ForecastRequest(
        horizon=ForecastHorizon.D30,
        product_ids=[pair.product_id],
        store_ids=[pair.store_id],
    ))
    print()
    print("valid request  :", ok.summary())

    refused = service.forecast(ForecastRequest(
        horizon=ForecastHorizon.D90,
        product_ids=[pair.product_id],
        store_ids=[pair.store_id],
        as_of_date=date(2025, 12, 15),
    ))
    print()
    print("beyond the calendar:")
    print(f"  status      : {refused.status}")
    print(f"  error_code  : {refused.error_code}")
    print(f"  recoverable : {refused.recoverable}")
    print(f"  message     : {refused.message}")

The refusal is **recoverable** and it names the latest as-of that would work. That distinction matters for Step 16: an agent can re-plan onto a valid request, whereas a bare "failed" leaves it with nothing to do but give up.

The alternative — assume no promotions are planned and carry the last price forward — produces a number that is systematically low and completely indistinguishable from a real forecast.

## 4. Interval width grows with horizon

In [ ]:
if service.is_available:
    pair = service.model.pairs.iloc[0]
    detail = service.model.predict_detail(
        horizon=ForecastHorizon.D90,
        product_ids=[pair.product_id],
        store_ids=[pair.store_id],
    )
    frame = detail.frame.copy()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(frame[TARGET_DATE], frame["predicted_units"], label="forecast", linewidth=1.3)
    if "lower_bound" in frame.columns:
        ax.fill_between(
            frame[TARGET_DATE], frame["lower_bound"], frame["upper_bound"],
            alpha=0.25, label="90% interval",
        )
    ax.set_title(f"90-day forecast path: {pair.product_id} @ {pair.store_id}")
    ax.set_ylabel("units")
    ax.legend()
    plt.tight_layout()
    plt.show()

    if "lower_bound" in frame.columns:
        frame["width_ratio"] = (
            (frame["upper_bound"] - frame["lower_bound"]) / frame["predicted_units"].clip(lower=1)
        )
        by_bucket = frame.groupby(
            frame["horizon_step"].map(config.intervals.bucket_for), observed=True
        )["width_ratio"].mean()
        print("Mean interval width relative to the forecast:")
        print(by_bucket.to_string())

Note the smoothness of the path. That is the payoff from drawing horizon steps at random during training rather than from a fixed grid — a grid makes the model's splits on `horizon_step` piecewise-constant, and the path comes out as a visible staircase.

If the width ratios above are flat across buckets, it means each bucket fell back to the pooled quantile because there were too few calibration points — which happens at smoke scale and resolves at full scale.

## 5. What the agent actually receives (§39)

In [ ]:
import json

from app.tools.forecasting_tool import ForecastingTool

if service.is_available:
    tool = ForecastingTool(service)
    pair = service.model.pairs.iloc[0]

    result = tool.run({
        "product_id": pair.product_id,
        "store_id": pair.store_id,
        "forecast_horizon": 30,
        "include_daily": False,
    })

    payload = result.for_llm()
    print(json.dumps(payload, indent=2, default=str)[:2000])

In [ ]:
if service.is_available:
    print("ASSUMPTIONS\n")
    for item in result.assumptions:
        print(f"  - {item}")
    print("\nWARNINGS\n")
    for item in result.warnings or ["(none)"]:
        print(f"  - {item}")

    # And a refusal, as the agent would see it.
    bad = tool.run({"forecast_horizon": 45})
    print("\nUNSUPPORTED HORIZON\n")
    print(f"  code        : {bad.error.code}")
    print(f"  recoverable : {bad.error.recoverable}")
    print(f"  message     : {bad.error.message}")
    print(f"  detail      : {bad.error.detail}")

Three properties worth noting in that output:

- **`confidence` is measured interval coverage**, not a mood. Either it is the calibrated nominal level or it is absent — there is no third option where a plausible-looking number appears because the field exists.
- **No internals leak.** No mention of LightGBM, parquet, MLflow or feature engineering. The agent asks a business question and gets a business answer with provenance.
- **An unsupported horizon is refused rather than rounded.** The model is calibrated at 7/14/30/90; serving 45 days as 30 would return a number whose interval does not describe it, and the agent would have no way to know.

---

## Summary

| Check | Why it matters |
|---|---|
| Mutation test | Corrupting the future must not change training features |
| **Planted bug** | Proves the mutation test can actually fail |
| Train/serve equivalence | The two feature paths must agree; this already caught two real defects |
| Error grows with horizon | The strongest behavioural evidence the join is right |
| Refusal past the calendar | Better than a forecast built on an unstated assumption |
| Measured coverage | Separates a real interval from a fabricated confidence score |

### The limit worth restating

This is a **demand** forecast, and it is **predictive**. It says what is likely given the planned prices and promotions. It does not say what a promotion *caused* — that is the uplift model's question, and conflating the two is how a seasonal peak gets credited to whatever campaign happened to be running.

**Next:** Step 6 builds promotion uplift on top of this.